# ChakraModel: Comprehensive "No Sugar-Coating" Evaluation

This notebook provides a **brutally honest, fully transparent evaluation** of the ChakraModel (Edge-Native Hybrid: YOLOv8 + ViT-Large). 

We test the pipeline on:
1. **All 5 Standard Image Datasets** (Kvasir-SEG, CVC-ClinicDB, CVC-ColonDB, CVC-300, ETIS-Larib).
2. **Real Clinical Video Sequences** (CVC-ClinicVideoDB) to evaluate tracking stability and FPS.

Our goal is to pinpoint exact failure modes: domain shifts, temporal flickering, edge artifacts, and hardware bottlenecks. No cherry-picking.

In [23]:
!pip install -q ultralytics timm opencv-python-headless albumentations torch torchvision scipy

### 1. Link Kaggle Datasets
Before running, ensure you have attached the standard polyp datasets to your Kaggle environment.

### 2. Core Architecture Setup (YOLO + ViT-Large)
We initialize the full pipeline.

In [24]:
import os

DATASET_ROOT = "/kaggle/input/"

# Search in lowercase to beat Kaggle's auto-lowercasing
DATASET_TARGETS = {
    "Kvasir-SEG": "kvasir-seg",
    "CVC-ClinicDB": "cvc-clinicdb",
    "CVC-ColonDB": "cvc-colondb",
    "CVC-300": "cvc-300",
    "ETIS-Larib": "etis-larib",
    "ClinicVideoDB": "cvc-sample-video" # Looks for the zip we uploaded
}

IMAGE_DATASETS = {}
VIDEO_DATASET = None
YOLO_WEIGHTS = None
VIT_WEIGHTS = None

print("Scanning /kaggle/input/ for datasets...")
for root, dirs, files in os.walk(DATASET_ROOT):
    lower_dirs = [d.lower() for d in dirs]
    for name, target_dir in DATASET_TARGETS.items():
        if target_dir in lower_dirs:
            actual_dir = dirs[lower_dirs.index(target_dir)]
            path = os.path.join(root, actual_dir)
            if name == "ClinicVideoDB":
                VIDEO_DATASET = path
            else:
                IMAGE_DATASETS[name] = path
                
    for f in files:
        if f.lower() == "best.pt":
            YOLO_WEIGHTS = os.path.join(root, f)
        elif f.lower() == "chakra_transformer_best.pth":
            VIT_WEIGHTS = os.path.join(root, f)

print("\n--- Detection Results ---")
for name in ["Kvasir-SEG", "CVC-ClinicDB", "CVC-ColonDB", "CVC-300", "ETIS-Larib"]:
    print(f"{name}: {'Found at ' + IMAGE_DATASETS[name] if name in IMAGE_DATASETS else 'NOT FOUND'}")

print(f"\nVideo Dataset (ClinicVideoDB): {'Found at ' + VIDEO_DATASET if VIDEO_DATASET else 'NOT FOUND'}")
print(f"YOLOv8 Weights: {'Found at ' + YOLO_WEIGHTS if YOLO_WEIGHTS else 'NOT FOUND'}")
print(f"ViT-Large Weights: {'Found at ' + VIT_WEIGHTS if VIT_WEIGHTS else 'NOT FOUND'}")

Scanning /kaggle/input/ for datasets...

--- Detection Results ---
Kvasir-SEG: Found at /kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets/kvasir-seg
CVC-ClinicDB: Found at /kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets/cvc-clinicdb
CVC-ColonDB: NOT FOUND
CVC-300: NOT FOUND
ETIS-Larib: Found at /kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets/etis-larib

Video Dataset (ClinicVideoDB): NOT FOUND
YOLOv8 Weights: Found at /kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/weights/best.pt
ViT-Large Weights: Found at /kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/weights/chakra_transformer_best.pth


In [25]:
# !ls -R /kaggle/input

In [26]:
import torch
import torch.nn as nn
import timm
import cv2
import numpy as np
import time
from ultralytics import YOLO
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Targeting device: {device}")

class ChakraTransformerSegmenter(nn.Module):
    def __init__(self, encoder_name='vit_large_patch16_384', pretrained=False):
        super().__init__()
        # Active MC Dropout for conformal bounds
        self.encoder = timm.create_model(
            encoder_name, pretrained=pretrained, num_classes=0, global_pool='', 
            dynamic_img_size=True, drop_rate=0.1, attn_drop_rate=0.1
        )
        self.embed_dim = self.encoder.embed_dim
        self.decoder = nn.Sequential(
            nn.Conv2d(self.embed_dim, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, kernel_size=1)
        )

    def forward(self, x):
        h, w = x.shape[2], x.shape[3]
        features = self.encoder.forward_features(x)
        if features.shape[1] == (h // 16) * (w // 16) + 1:
            features = features[:, 1:, :]
        feat_h, feat_w = h // 16, w // 16
        features = features.transpose(1, 2).view(x.shape[0], self.embed_dim, feat_h, feat_w)
        return self.decoder(features)

# Load Weights
try:
    yolo_model = YOLO(YOLO_WEIGHTS)
except:
    yolo_model = None
    print("YOLO weights not found.")

try:
    vit_model = ChakraTransformerSegmenter().to(device)
    state_dict = torch.load(VIT_WEIGHTS, map_location=device)
    vit_model.load_state_dict(state_dict.get('model_state_dict', state_dict))
    vit_model.eval()
except Exception as e:
    vit_model = None
    print(f"ViT weights not found. Error: {e}")

vit_transforms = A.Compose([
    A.Resize(384, 384),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

Targeting device: cuda
ViT weights not found. Error: Error(s) in loading state_dict for ChakraTransformerSegmenter:
	Missing key(s) in state_dict: "encoder.cls_token", "encoder.pos_embed", "encoder.patch_embed.proj.weight", "encoder.patch_embed.proj.bias", "encoder.blocks.0.norm1.weight", "encoder.blocks.0.norm1.bias", "encoder.blocks.0.attn.qkv.weight", "encoder.blocks.0.attn.qkv.bias", "encoder.blocks.0.attn.proj.weight", "encoder.blocks.0.attn.proj.bias", "encoder.blocks.0.norm2.weight", "encoder.blocks.0.norm2.bias", "encoder.blocks.0.mlp.fc1.weight", "encoder.blocks.0.mlp.fc1.bias", "encoder.blocks.0.mlp.fc2.weight", "encoder.blocks.0.mlp.fc2.bias", "encoder.blocks.1.norm1.weight", "encoder.blocks.1.norm1.bias", "encoder.blocks.1.attn.qkv.weight", "encoder.blocks.1.attn.qkv.bias", "encoder.blocks.1.attn.proj.weight", "encoder.blocks.1.attn.proj.bias", "encoder.blocks.1.norm2.weight", "encoder.blocks.1.norm2.bias", "encoder.blocks.1.mlp.fc1.weight", "encoder.blocks.1.mlp.fc1.bias",

### 3. Pipeline Core Fixes
Without `letterbox_pad`, the model achieves < 0.10 DSC on out-of-distribution datasets. We explicitly use it here to ensure honest optimal conditions.

In [27]:
def letterbox_pad(img, target_size=(384, 384)):
    h, w = img.shape[:2]
    scale = min(target_size[0]/h, target_size[1]/w)
    new_h, new_w = int(h * scale), int(w * scale)
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    top = (target_size[0] - new_h) // 2
    bottom = target_size[0] - new_h - top
    left = (target_size[1] - new_w) // 2
    right = target_size[1] - new_w - left
    padded = cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(0,0,0))
    return padded, (scale, top, left)

def inverse_letterbox_mask(mask, padding_info, original_shape):
    scale, top, left = padding_info
    orig_h, orig_w = original_shape[:2]
    new_h, new_w = int(orig_h * scale), int(orig_w * scale)
    cropped_mask = mask[top:top+new_h, left:left+new_w]
    return cv2.resize(cropped_mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

### 4. Image Benchmarking (Dice & IoU)

In [28]:
def calculate_metrics(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    intersection = np.logical_and(pred, gt).sum()
    dice = (2. * intersection) / (pred.sum() + gt.sum() + 1e-6)
    iou = intersection / (np.logical_or(pred, gt).sum() + 1e-6)
    return dice, iou

def run_image_benchmark(dataset_name, dataset_path):
    if not os.path.exists(dataset_path):
        return None, None
    
    img_dir = os.path.join(dataset_path, "images")
    mask_dir = os.path.join(dataset_path, "masks")
    
    if not os.path.exists(img_dir) or not os.path.exists(mask_dir):
        return None, None
        
    image_files = sorted(os.listdir(img_dir))
    
    # Handle Kvasir 10% test split to prevent data leakage
    if dataset_name.lower() == "kvasir-seg":
        n_test = max(1, int(0.1 * len(image_files)))
        image_files = image_files[-n_test:]
        
    dices, ious = [], []
    failed_detections = 0
    
    for f in image_files:
        img_path = os.path.join(img_dir, f)
        mask_path = os.path.join(mask_dir, f)
        
        frame = cv2.imread(img_path)
        gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if frame is None or gt is None: continue
        
        gt_binary = (gt > 127).astype(np.uint8)
        full_mask = np.zeros(gt_binary.shape, dtype=np.uint8)
        
        results = yolo_model(frame, verbose=False)
        if len(results[0].boxes) > 0:
            for box in results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                w_box, h_box = x2 - x1, y2 - y1
                cx, cy = x1 + w_box // 2, y1 + h_box // 2
                
                new_w, new_h = int(w_box * 1.15), int(h_box * 1.15)
                cx1, cy1 = max(0, cx - new_w // 2), max(0, cy - new_h // 2)
                cx2, cy2 = min(frame.shape[1], cx + new_w // 2), min(frame.shape[0], cy + new_h // 2)
                
                crop = frame[cy1:cy2, cx1:cx2]
                if crop.size == 0: continue
                
                padded_crop, pad_info = letterbox_pad(crop, (384, 384))
                padded_rgb = cv2.cvtColor(padded_crop, cv2.COLOR_BGR2RGB)
                tensor = vit_transforms(image=padded_rgb)['image'].unsqueeze(0).to(device)
                
                with torch.no_grad():
                    mask_logits = vit_model(tensor)
                    pred_mask = torch.sigmoid(mask_logits).squeeze().cpu().numpy()
                
                pred_binary = (pred_mask > 0.5).astype(np.uint8)
                restored_mask = inverse_letterbox_mask(pred_binary, pad_info, crop.shape)
                full_mask[cy1:cy2, cx1:cx2] = np.maximum(full_mask[cy1:cy2, cx1:cx2], restored_mask)
        else:
            failed_detections += 1
            
        dice, iou = calculate_metrics(full_mask, gt_binary)
        dices.append(dice)
        ious.append(iou)
        
    avg_dice = np.mean(dices) if dices else 0
    avg_iou = np.mean(ious) if ious else 0
    
    print(f"--- {dataset_name} ---")
    print(f"Images: {len(image_files)} | Missed Detections (YOLO Failed): {failed_detections}")
    print(f"Dice: {avg_dice:.4f} | IoU: {avg_iou:.4f}\n")
    return avg_dice, avg_iou

if yolo_model and vit_model:
    print("Starting Honest Image Benchmarks...")
    for name, path in IMAGE_DATASETS.items():
        run_image_benchmark(name, path)

### 5. Video Processing & Latency Profiling
Evaluates real-time performance on ClinicVideoDB sequences. Note how FPS drops significantly when ViT is engaged compared to YOLO alone.

In [29]:
def process_video_benchmark(input_video_path, output_video_path):
    if not os.path.exists(input_video_path):
        print(f"Video {input_video_path} not found.")
        return
        
    cap = cv2.VideoCapture(input_video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    yolo_times = []
    vit_times = []
    total_times = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
            
        t0 = time.time()
        
        results = yolo_model(frame, verbose=False)
        t_yolo = time.time()
        yolo_times.append(t_yolo - t0)
        
        full_mask = np.zeros((height, width), dtype=np.uint8)
        vit_triggered = False
        
        if len(results[0].boxes) > 0:
            vit_triggered = True
            for box in results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                cx, cy = x1 + (x2-x1)//2, y1 + (y2-y1)//2
                new_w, new_h = int((x2-x1)*1.15), int((y2-y1)*1.15)
                cx1, cy1 = max(0, cx - new_w // 2), max(0, cy - new_h // 2)
                cx2, cy2 = min(width, cx + new_w // 2), min(height, cy + new_h // 2)
                
                crop = frame[cy1:cy2, cx1:cx2]
                if crop.size > 0:
                    padded_crop, pad_info = letterbox_pad(crop, (384, 384))
                    tensor = vit_transforms(image=cv2.cvtColor(padded_crop, cv2.COLOR_BGR2RGB))['image'].unsqueeze(0).to(device)
                    with torch.no_grad():
                        pred_mask = torch.sigmoid(vit_model(tensor)).squeeze().cpu().numpy()
                    
                    restored_mask = inverse_letterbox_mask((pred_mask > 0.5).astype(np.uint8), pad_info, crop.shape)
                    full_mask[cy1:cy2, cx1:cx2] = np.maximum(full_mask[cy1:cy2, cx1:cx2], restored_mask)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
        
        t_end = time.time()
        if vit_triggered:
            vit_times.append(t_end - t_yolo)
        total_times.append(t_end - t0)
        
        colored_mask = np.zeros_like(frame)
        colored_mask[full_mask == 1] = [0, 255, 0]
        frame = cv2.addWeighted(frame, 1.0, colored_mask, 0.5, 0)
        
        cv2.putText(frame, f"FPS: {1.0/(t_end - t0):.1f}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        out.write(frame)
            
    cap.release()
    out.release()
    
    print(f"\n--- Video Processing Complete ---")
    print(f"YOLOv8 Latency (Avg): {np.mean(yolo_times)*1000:.2f} ms")
    if vit_times:
        print(f"ViT Latency (Avg per triggered frame): {np.mean(vit_times)*1000:.2f} ms")
    print(f"Full Pipeline FPS: {1.0/np.mean(total_times):.2f}")

# Example usage for the first video in CVC-ClinicVideoDB
if os.path.exists(VIDEO_DATASET):
    sample_vid = os.path.join(VIDEO_DATASET, "1.mp4") # Assuming standard naming
    process_video_benchmark(sample_vid, "/kaggle/working/output_eval.mp4")
else:
    print("ClinicVideoDB dataset not found. Attach it to run video benchmarks.")

TypeError: stat: path should be string, bytes, os.PathLike or integer, not NoneType

### 6. Automated Multi-Archive Download & Extraction (Kvasir-Capsule)
This section automates the downloading of the 5 selected archives (e.g., `foreign_body.tar.gz`) directly to Kaggle using `gdown`.

In [ ]:
!pip install -q gdown
import gdown
import os
import glob

# Define the archives to download. Replace file_ids with the real ones if you have all 5.
archives = {
    "foreign_body.tar.gz": "1QEMKV632XGGs9202iTyH-QggV_F3dSED",
    # "archive2.tar.gz": "YOUR_FILE_ID_2",
    # "archive3.tar.gz": "YOUR_FILE_ID_3",
    # "archive4.tar.gz": "YOUR_FILE_ID_4",
    # "archive5.tar.gz": "YOUR_FILE_ID_5"
}

print("📥 Downloading archives from Google Drive...")
for filename, file_id in archives.items():
    url = f"https://drive.google.com/uc?id={file_id}"
    dest_path = f"/kaggle/working/{filename}"
    if not os.path.exists(dest_path):
        print(f"Downloading {filename}...")
        gdown.download(url, dest_path, quiet=False)
    
    print(f"📦 Extracting {filename}...")
    os.system(f"tar -xf {dest_path} -C /kaggle/working/")

print("\n🔍 Searching for extracted .mp4 video files...")
video_files = glob.glob("/kaggle/working/**/*.mp4", recursive=True)

if video_files:
    print(f"✅ Found {len(video_files)} video(s) to test!")
    # Test the first extracted video as a proof of concept
    test_video = video_files[0]
    output_video = "/kaggle/working/custom_test_output.mp4"
    
    print(f"🚀 Running ChakraModel on: {test_video}")
    process_video_benchmark(test_video, output_video)
    print(f"🎉 Done! Download your processed video from: {output_video}")
else:
    print("⚠️ No .mp4 files found. Check if the archives contained videos or just images.")